# ETL Gold - IT Support

**Camada Gold**: Dados agregados e prontos para consumo em dashboards e relatórios.

Este notebook cria as seguintes tabelas a partir da camada Silver:
1. **gold_support_kpis** — KPIs gerais de suporte (total tickets, tempo médio, SLA, satisfação)
2. **gold_tickets_by_country** — Métricas agregadas por país
3. **gold_agent_performance** — Performance individual de cada atendente
4. **gold_sla_compliance** — Conformidade de SLA por prioridade e país

**Source/Target:** Configurados via `00_Config` (catalog e schema)

In [0]:
# --------------------------------------------------
# Configuração do ETL (herda catalog/schema do 00_Config)
# --------------------------------------------------
# MAGIC %run ./00_Config

In [0]:
%sql
-- --------------------------------------------------
-- Visualização das tabelas Silver (fonte)
-- --------------------------------------------------
SELECT 'silver_tickets_full' AS tabela, COUNT(*) AS registros FROM silver_tickets_full
UNION ALL
SELECT 'silver_comments_enriched', COUNT(*) FROM silver_comments_enriched;

In [0]:
%sql
-- --------------------------------------------------
-- gold_support_kpis
-- KPIs gerais consolidados do suporte
-- --------------------------------------------------
CREATE OR REPLACE TABLE gold_support_kpis AS
SELECT
    COUNT(ticket_id) AS total_tickets,
    COUNT(DISTINCT agent_name) AS total_agents,
    ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(first_response_hours), 2) AS avg_first_response_hours,
    ROUND(AVG(survey_results), 2) AS avg_survey_score,
    ROUND(SUM(CASE WHEN sla_resolution_met = TRUE THEN 1 ELSE 0 END) / COUNT(ticket_id) * 100, 2) AS sla_resolution_pct,
    ROUND(SUM(CASE WHEN sla_first_response_met = TRUE THEN 1 ELSE 0 END) / COUNT(ticket_id) * 100, 2) AS sla_first_response_pct,
    SUM(CASE WHEN status = 'Open' THEN 1 ELSE 0 END) AS open_tickets,
    SUM(CASE WHEN status = 'Closed' THEN 1 ELSE 0 END) AS closed_tickets,
    ROUND(AVG(agent_interactions), 2) AS avg_interactions_per_ticket,
    CURRENT_TIMESTAMP() AS etl_timestamp
FROM silver_tickets_full;

In [0]:
%sql
-- --------------------------------------------------
-- Preview da tabela gold_support_kpis
-- --------------------------------------------------
SELECT * FROM gold_support_kpis;

In [0]:
%sql
-- --------------------------------------------------
-- gold_tickets_by_country
-- Métricas agregadas por país
-- --------------------------------------------------
CREATE OR REPLACE TABLE gold_tickets_by_country AS
SELECT
    country,
    COUNT(ticket_id) AS total_tickets,
    ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(survey_results), 2) AS avg_survey_score,
    ROUND(SUM(CASE WHEN sla_resolution_met = TRUE THEN 1 ELSE 0 END) / COUNT(ticket_id) * 100, 2) AS sla_compliance_pct,
    COUNT(DISTINCT agent_name) AS agents_count,
    SUM(CASE WHEN priority = 'Critical' THEN 1 ELSE 0 END) AS critical_tickets,
    SUM(CASE WHEN priority = 'High' THEN 1 ELSE 0 END) AS high_tickets,
    CURRENT_TIMESTAMP() AS etl_timestamp
FROM silver_tickets_full
GROUP BY country
ORDER BY total_tickets DESC;

In [0]:
%sql
-- --------------------------------------------------
-- Preview da tabela gold_tickets_by_country
-- --------------------------------------------------
SELECT * FROM gold_tickets_by_country;

In [0]:
%sql
-- --------------------------------------------------
-- gold_agent_performance
-- Métricas individuais por atendente
-- --------------------------------------------------
CREATE OR REPLACE TABLE gold_agent_performance AS
SELECT
    agent_name,
    agent_group,
    COUNT(ticket_id) AS total_tickets,
    ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(first_response_hours), 2) AS avg_first_response_hours,
    ROUND(AVG(survey_results), 2) AS avg_survey_score,
    ROUND(SUM(CASE WHEN sla_resolution_met = TRUE THEN 1 ELSE 0 END) / COUNT(ticket_id) * 100, 2) AS sla_compliance_pct,
    ROUND(AVG(agent_interactions), 2) AS avg_interactions,
    COUNT(DISTINCT country) AS countries_served,
    CURRENT_TIMESTAMP() AS etl_timestamp
FROM silver_tickets_full
WHERE agent_name IS NOT NULL
GROUP BY agent_name, agent_group
ORDER BY total_tickets DESC;

In [0]:
%sql
-- --------------------------------------------------
-- Preview da tabela gold_agent_performance
-- --------------------------------------------------
SELECT * FROM gold_agent_performance LIMIT 10;

In [0]:
%sql
-- --------------------------------------------------
-- gold_sla_compliance
-- Conformidade de SLA por prioridade e país
-- --------------------------------------------------
CREATE OR REPLACE TABLE gold_sla_compliance AS
SELECT
    priority,
    country,
    COUNT(ticket_id) AS total_tickets,
    SUM(CASE WHEN sla_resolution_met = TRUE THEN 1 ELSE 0 END) AS tickets_within_sla,
    SUM(CASE WHEN sla_resolution_met = FALSE THEN 1 ELSE 0 END) AS tickets_breached_sla,
    ROUND(SUM(CASE WHEN sla_resolution_met = TRUE THEN 1 ELSE 0 END) / COUNT(ticket_id) * 100, 2) AS sla_compliance_pct,
    ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours,
    ROUND(AVG(survey_results), 2) AS avg_survey_score,
    CURRENT_TIMESTAMP() AS etl_timestamp
FROM silver_tickets_full
GROUP BY priority, country
ORDER BY priority, total_tickets DESC;

In [0]:
%sql
-- --------------------------------------------------
-- Validação: Contagem das tabelas Gold criadas
-- --------------------------------------------------
SELECT 'gold_support_kpis' AS tabela, COUNT(*) AS registros FROM gold_support_kpis
UNION ALL
SELECT 'gold_tickets_by_country', COUNT(*) FROM gold_tickets_by_country
UNION ALL
SELECT 'gold_agent_performance', COUNT(*) FROM gold_agent_performance
UNION ALL
SELECT 'gold_sla_compliance', COUNT(*) FROM gold_sla_compliance;